In [9]:
# =========================
# 0) IMPORT
# =========================
import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE

In [10]:
# =========================
# 1) LOAD DATA
# =========================
DATA_PATH = "diabetes_frankurt_germany_1.xlsx"
raw = pd.read_excel(DATA_PATH)

print("Shape awal:", raw.shape)
raw.head()

Shape awal: (2000, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,2,138.0,62.0,35,0.0,33.6,0.127,47,1
1,0,84.0,82.0,31,125.0,38.2,0.233,23,0
2,0,145.0,72.0,29,126.0,44.2,0.630,31,1
3,0,135.0,68.0,42,250.0,42.3,0.365,24,1
4,1,139.0,62.0,41,480.0,40.7,0.536,21,0


In [11]:
# =========================
# 1) LOAD DATA
# =========================
DATA_PATH = "diabetes_frankurt_germany_1.xlsx"
data = pd.read_excel(DATA_PATH)
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,2,138.0,62.0,35,0.0,33.6,0.127,47,1
1,0,84.0,82.0,31,125.0,38.2,0.233,23,0
2,0,145.0,72.0,29,126.0,44.2,0.630,31,1
3,0,135.0,68.0,42,250.0,42.3,0.365,24,1
4,1,139.0,62.0,41,480.0,40.7,0.536,21,0


In [12]:
# =========================
# 2) PENGECEKAN (sama seperti notebook lama)
#    - cek duplikat
#    - cek nol per kolom
#    - hitung persentase_baris_nol (dipakai untuk keputusan imputasi)
# =========================

# cek duplikat
jumlah_duplikat = data.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

df_duplikat = data[data.duplicated(keep=False)].copy()
df_duplikat.insert(0, "No", df_duplikat.index)
df_duplikat.head()

Jumlah data duplikat: 8


,No,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
494,494,3,80.0,72.0,29,126.0,32.4,0.2,22,0
495,495,6,166.0,74.0,0,0.0,26.6,0.3,66,0
868,868,3,82.0,70.0,29,126.0,21.1,0.4,25,0
958,958,4,99.0,72.0,17,126.0,25.6,0.3,28,0
961,961,2,89.0,90.0,30,0.0,33.5,0.3,42,0


In [13]:
# Cek nilai 0 (sama seperti notebook lama)
print("\nNilai 0 pada setiap kolom:")
jumlah_nol = {}
for kolom in data.columns:
    if kolom != "Outcome":
        hitung_nol = (data[kolom] == 0).sum()
        jumlah_nol[kolom] = hitung_nol
        print(f"{kolom}: {hitung_nol}")

# Total baris yang memiliki minimal satu nilai 0 (selain Outcome dan Pregnancies)
fitur_dengan_nol = data.drop(["Outcome", "Pregnancies"], axis=1) == 0
total_baris_nol = fitur_dengan_nol.any(axis=1).sum()
persentase_baris_nol = (total_baris_nol / len(data)) * 100

print(f"Total baris yang memiliki minimal satu nilai 0: {total_baris_nol}")
print(f"Persentase baris dengan nilai 0: {persentase_baris_nol:.2f}%")


Nilai 0 pada setiap kolom:
Pregnancies: 301
Glucose: 4
BloodPressure: 32
SkinThickness: 176
Insulin: 316
BMI: 13
DiabetesPedigreeFunction: 0
Age: 0
Total baris yang memiliki minimal satu nilai 0: 320
Persentase baris dengan nilai 0: 16.00%


In [14]:
# =========================
# 3) PREPROCESSING (sama seperti notebook lama)
#    - hapus duplikat
#    - imputasi KNN jika persentase_baris_nol > 10 (0 dianggap missing untuk kolom tertentu)
# =========================

# hapus duplikat
if jumlah_duplikat > 0:
    data = data.drop_duplicates().reset_index(drop=True)
    print(f"Jumlah data duplikat setelah penghapusan: {data.duplicated().sum()}")

# imputasi / drop row (sesuai notebook lama)
data_temp = data.copy()
kolom_hilang = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

if persentase_baris_nol > 10:
    data_temp[kolom_hilang] = data_temp[kolom_hilang].replace(0, np.nan)

    X_untuk_imputasi = data_temp.drop(columns=["Outcome"])
    y = data_temp["Outcome"]

    imputer_knn = KNNImputer(n_neighbors=5)
    X_terimputasi = imputer_knn.fit_transform(X_untuk_imputasi)

    data = pd.DataFrame(X_terimputasi, columns=X_untuk_imputasi.columns)
    data["Outcome"] = y.to_numpy()

    print(f"Jumlah missing values setelah imputasi:\n{data.isnull().sum()}")
else:
    data = data_temp.dropna(subset=kolom_hilang).reset_index(drop=True)
    print(f"Jumlah data setelah menghapus baris dengan nilai 0: {len(data)}")

data.head()

Jumlah data duplikat setelah penghapusan: 0
Jumlah missing values setelah imputasi:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,2.0,138.0,62.0,35.0,126.0,33.6,0.127,47.0,1
1,0.0,84.0,82.0,31.0,125.0,38.2,0.233,23.0,0
2,0.0,145.0,72.0,29.0,126.0,44.2,0.630,31.0,1
3,0.0,135.0,68.0,42.0,250.0,42.3,0.365,24.0,1
4,1.0,139.0,62.0,41.0,480.0,40.7,0.536,21.0,0


In [15]:
# =========================
# 4) LOFO + STACKING meta-KNN
#    IMPORTANT: normalisasi fit_transform SEBELUM split (meniru notebook lama)
# =========================

# best params base (ambil dari notebook lama kamu)
BEST_BASE_PARAMS = {
    "knn": dict(n_neighbors=9, weights="distance", metric="manhattan"),
    "lr":  dict(C=0.03359818286283781, penalty="l2", solver="liblinear"),
    "svm": dict(C=1000, gamma="scale", kernel="rbf"),
    "ann": dict(max_iter=1000, hidden_layer_sizes=(50, 50), activation="relu", alpha=0.0001),
}

# best params meta KNN (stacking terbaik kamu)
BEST_META_KNN_PARAMS = dict(n_neighbors=13, weights="uniform", metric="manhattan")

RANDOM_STATE = 42
N_SPLITS = 5


def make_base_models():
    knn = KNeighborsClassifier(**BEST_BASE_PARAMS["knn"])
    lr  = LogisticRegression(random_state=RANDOM_STATE, **BEST_BASE_PARAMS["lr"])
    svm = SVC(random_state=RANDOM_STATE, probability=True, **BEST_BASE_PARAMS["svm"])
    ann = MLPClassifier(random_state=RANDOM_STATE, **BEST_BASE_PARAMS["ann"])
    return [("LogReg", lr), ("KNN", knn), ("SVM", svm), ("ANN", ann)]


def make_meta_model():
    return KNeighborsClassifier(**BEST_META_KNN_PARAMS)


def build_meta_features_oof(base_models, X_train, y_train, cv):
    feats = []
    for _, model in base_models:
        proba_oof = cross_val_predict(
            model,
            X_train,
            y_train,
            cv=cv,
            method="predict_proba",
            n_jobs=-1
        )
        feats.append(proba_oof)   # (n,2)
    return np.hstack(feats)       # (n,8)


def build_meta_features_test(fitted_base_models, X_test):
    feats = []
    for _, model in fitted_base_models:
        feats.append(model.predict_proba(X_test))  # (n,2)
    return np.hstack(feats)                       # (n,8)


def eval_stacking_meta_knn_like_old_pipeline(data_df, drop_feature=None):
    # ambil X,y
    X = data_df.drop(columns=["Outcome"]).copy()
    y = data_df["Outcome"].copy()

    # hapus 1 fitur (LOFO)
    if drop_feature is not None:
        X = X.drop(columns=[drop_feature])

    # NORMALISASI dulu, baru split (ini yang bikin match notebook lama)
    scaler = MinMaxScaler()
    X_norm = scaler.fit_transform(X)
    X_norm = pd.DataFrame(X_norm, columns=X.columns)

    # split
    X_train, X_test, y_train, y_test = train_test_split(
        X_norm, y, test_size=0.3, random_state=RANDOM_STATE
    )

    # SMOTE (sama seperti notebook lama)
    jumlah_per_kelas = y_train.value_counts()
    jumlah_kelas_mayoritas = jumlah_per_kelas.max()
    jumlah_kelas_minoritas = jumlah_per_kelas.min()
    IR = jumlah_kelas_mayoritas / jumlah_kelas_minoritas

    if IR > 1.5:
        smote = SMOTE(random_state=RANDOM_STATE)
        X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    else:
        X_train_smote, y_train_smote = X_train, y_train

    # kfold sama seperti notebook lama
    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # base models
    base_models = make_base_models()

    # meta-train features (OOF proba)
    X_meta_latih = build_meta_features_oof(base_models, X_train_smote, y_train_smote, cv=kfold)

    # fit base models di full X_train_smote untuk meta-test features
    fitted_base = []
    for name, model in base_models:
        model.fit(X_train_smote, y_train_smote)
        fitted_base.append((name, model))

    X_meta_uji = build_meta_features_test(fitted_base, X_test)

    # meta learner KNN
    meta = make_meta_model()
    meta.fit(X_meta_latih, y_train_smote)
    y_pred = meta.predict(X_meta_uji)

    return {
        "Fitur_dihapus": "Tidak ada" if drop_feature is None else drop_feature,
        "Jumlah_fitur": int(X.shape[1]),
        "IR_train": float(IR),
        "Akurasi": float(accuracy_score(y_test, y_pred)),
        "Presisi": float(precision_score(y_test, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
    }


# =========================
# 5) JALANKAN BASELINE + LOFO
# =========================
fitur_list = [c for c in data.columns if c != "Outcome"]

results = []
results.append(eval_stacking_meta_knn_like_old_pipeline(data, drop_feature=None))  # baseline (tidak ada dihapus)

for f in fitur_list:
    results.append(eval_stacking_meta_knn_like_old_pipeline(data, drop_feature=f))

df_lofo = pd.DataFrame(results)

# delta vs baseline (supaya terlihat fitur paling berpengaruh)
baseline = df_lofo.loc[df_lofo["Fitur_dihapus"] == "Tidak ada"].iloc[0]
for col in ["Akurasi", "Presisi", "Recall", "F1"]:
    df_lofo[f"Delta_{col}"] = df_lofo[col] - baseline[col]

df_lofo.sort_values("Delta_Akurasi").reset_index(drop=True)

,Fitur_dihapus,Jumlah_fitur,IR_train,Akurasi,Presisi,Recall,F1,Delta_Akurasi,Delta_Presisi,Delta_Recall,Delta_F1
0,Pregnancies,7,1.898129,0.918060,0.859155,0.905941,0.881928,-0.021739,-0.025104,-0.039604,-0.031948
1,BMI,7,1.898129,0.928094,0.866359,0.930693,0.897375,-0.011706,-0.017900,-0.014851,-0.016501
2,Insulin,7,1.898129,0.929766,0.870370,0.930693,0.899522,-0.010033,-0.013889,-0.014851,-0.014354
3,SkinThickness,7,1.898129,0.933110,0.871560,0.940594,0.904762,-0.006689,-0.012700,-0.004950,-0.009114
4,DiabetesPedigreeFunction,7,1.898129,0.933110,0.889423,0.915842,0.902439,-0.006689,0.005164,-0.029703,-0.011437
5,Glucose,7,1.898129,0.934783,0.882629,0.930693,0.906024,-0.005017,-0.001630,-0.014851,-0.007852
6,BloodPressure,7,1.898129,0.939799,0.887850,0.940594,0.913462,0.000000,0.003591,-0.004950,-0.000414
7,Tidak ada,8,1.898129,0.939799,0.884259,0.945545,0.913876,0.000000,0.000000,0.000000,0.000000
8,Age,7,1.898129,0.939799,0.899038,0.925743,0.912195,0.000000,0.014779,-0.019802,-0.001680


In [16]:
# =========================
# EXPORT HASIL LOFO KE EXCEL
# =========================
import pandas as pd

# versi ranking (fitur paling berpengaruh biasanya Delta_Akurasi paling negatif)
df_lofo_ranking = df_lofo.sort_values("Delta_Akurasi", ascending=True).reset_index(drop=True)

output_excel = "hasil_lofo_stacking_meta_knn.xlsx"

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    df_lofo.to_excel(writer, sheet_name="Semua_Hasil", index=False)
    df_lofo_ranking.to_excel(writer, sheet_name="Ranking", index=False)

print("Berhasil membuat:", output_excel)
print("Total baris:", len(df_lofo))

Berhasil membuat: hasil_lofo_stacking_meta_knn.xlsx
Total baris: 9
